# GEMMA 3N E4B-IT AUDIO TRAINING FOR TEKNOFEST 2025
### Turkish Telco Call Center Agent with Audio Understanding

In [ ]:
# STEP 1: Install Unsloth and dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade transformers datasets accelerate peft bitsandbytes trl

In [ ]:
# STEP 2: Load Gemma 3N E4B-IT Model
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-3n-e4b-it",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map="auto"
)

print("✅ Gemma 3N E4B-IT loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# STEP 3: Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True
)

print(f"✅ Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}M params")

In [ ]:
# STEP 4: Mount Google Drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Upload your dataset to Drive first!
# Expected path: /content/drive/MyDrive/teknofest/gemma3n_audio_training.jsonl

In [ ]:
# STEP 5: Load and prepare dataset
import json
import pandas as pd
from datasets import Dataset

def load_training_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load dataset
DATASET_PATH = '/content/drive/MyDrive/teknofest/gemma3n_audio_training.jsonl'
training_data = load_training_data(DATASET_PATH)
dataset = Dataset.from_pandas(pd.DataFrame(training_data))

# Split
train_test = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test['train']
eval_dataset = train_test['test']

print(f"Training: {len(train_dataset)}, Validation: {len(eval_dataset)}")

In [ ]:
# STEP 6: Format data for training
def format_prompt(example):
    context = json.loads(example['context'])
    output = json.loads(example['output'])
    
    history = ""
    if context:
        for turn in context[-4:]:
            if turn['type'] == 'customer':
                history += f"Müşteri: {turn['text']}\n"
            elif turn['type'] == 'agent':
                history += f"{turn['agent']}: {turn['text']}\n"
    
    prompt = f"""### Konuşma:
{history}
[Audio: {example['audio']}]

### Yanıt:
Agent: {output['agent']}
Tools: {', '.join(output['tools'])}
Response: {output['response']}"""
    
    return {"text": prompt}

train_dataset = train_dataset.map(format_prompt)
eval_dataset = eval_dataset.map(format_prompt)

In [ ]:
# STEP 7: Training configuration
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./gemma3n-telco",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    learning_rate=2e-4,
    fp16=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=max_seq_length
)

In [ ]:
# STEP 8: Train!
print("🚀 Starting training...")
trainer_stats = trainer.train()
print(f"✅ Training complete! Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# STEP 9: Save model
model.save_pretrained("gemma3n-telco-final")
tokenizer.save_pretrained("gemma3n-telco-final")

# Save to Drive
!cp -r gemma3n-telco-final /content/drive/MyDrive/teknofest/
print("✅ Model saved to Google Drive!")

In [ ]:
# STEP 10: Test the model
def test_model(audio_path, context_text):
    prompt = f"""### Konuşma:
{context_text}
[Audio: {audio_path}]

### Yanıt:"""
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return response.split("### Yanıt:")[-1].strip()

# Test
test_response = test_model(
    "test_audio.mp3",
    "Müşteri: eSIM'im çalışmıyor, yardım eder misiniz?"
)
print("Generated:", test_response)